In [0]:
%sql
-- ============================================================================
-- 6. mkt_dashboard_tickets_monthly  (grain: month x population_type, 3 rows/month)
--    PRODUCTION (incremental) on VisitDateId: loads every closed month after the
--    latest report_month in the table (Jan 2025 if the table is empty), once
--    that month's month-end subscriber snapshot exists (tickets_in_market needs it).
--    Cohorts: CC / Scene (Non-CC Scene only) / Total_Biz.
--    CC/Scene distinct_members + total_visits follow MOM notebook logic
--    (any theatre purchase, netted per member x day x location, net > 0).
--    Validated vs MOM July 2026 (CC within 0.1%; Scene tickets +1.2%, logic
--    identical -- attributed to post-MOM data refresh).
--    Query = final event-based tickets_credited version (Sep 24 chat).
--    Column list follows the SELECT's own order so every value lands in the
--    right column (the SELECT order differs from the table order).
-- ============================================================================
BEGIN
  DECLARE v_report_month DATE;
  DECLARE v_end          DATE;

  SET v_report_month = (
    SELECT COALESCE(add_months(MAX(report_month), 1), DATE'2025-01-01')
    FROM data_exploration_hub.marketing_hub.mkt_dashboard_tickets_monthly
  );
  SET v_end = (
    SELECT LEAST(
      (SELECT to_date(CAST(MAX(snapshot_date_id) AS STRING), 'yyyyMMdd')
         FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot),
      date_sub(current_date(), 1)
    )
  );

  WHILE last_day(v_report_month) <= v_end DO
    DELETE FROM data_exploration_hub.marketing_hub.mkt_dashboard_tickets_monthly
    WHERE report_month = v_report_month;

    INSERT INTO data_exploration_hub.marketing_hub.mkt_dashboard_tickets_monthly (
      report_month,
      population_type,
      distinct_members,
      total_visits,
      tickets_regular,
      tickets_upgrades,
      tickets_dbox,
      tickets_avx,
      tickets_imax,
      tickets_vip,
      tickets_alt,
      tickets_fourdx,
      tickets_screenx,
      tickets_threed,
      tickets_free,
      tickets_paid,
      tickets_free_redeemed,
      companion_tickets_redeemed,
      tickets_undiscounted,
      share_9_99_Free,
      share_10_99_Free,
      share_9_99_Companion,
      share_10_99_Companion,
      tickets_credited,
      tickets_in_market,
      share_9_99_Credited,
      share_10_99_Credited,
      share_9_99_InMarket,
      share_10_99_InMarket
    )
    WITH params AS (
        SELECT v_report_month AS report_month
    ),

    month_bounds AS (
        SELECT
            report_month                                   AS month_start,
            LAST_DAY(report_month)                          AS month_end,
            ADD_MONTHS(report_month, -1)                    AS prior_month_start,
            LAST_DAY(ADD_MONTHS(report_month, -1))          AS prior_month_end
        FROM params
    ),

    -- ----------------------------------------------------------------------------
    -- Ticket-line grain: EXH only, joined to session for format classification.
    -- Date dimension is joined on VisitDateId (not TransactionDateId) -- the
    -- whole table reports on visit month, matching the business definition.
    -- No CDE_ID filter here -- Total_Biz needs anonymous/unidentified ticket
    -- buyers included. CC/Scene branches apply CDE_ID IS NOT NULL themselves.
    -- ----------------------------------------------------------------------------
    ticket_events AS (
        SELECT
            t.CDE_ID                            AS cde_id,
            d.datefull                          AS datefull,
            t.CineClubFLAG                      AS cineclub_flag,
            t.SCENEFLAG                         AS scene_flag,
            tk.paid_attendance_inflag           AS paid_flag,
            t.CineClub_Discount_Type_ID         AS discount_type_id,
            t.Quantity                          AS quantity,
            s.Auditorium_Experience_ID          AS auditorium_experience_id,
            s.Performance_Experience_ID         AS performance_experience_id,
            s.Is_DBOXflag                       AS is_dbox_flag
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc
            ON t.LocationId = loc.LocationID
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d
            ON t.VisitDateId = d.dateid
        LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_session s
            ON t.sessionskey = s.sessionskey
        LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_ticket_type tk
            ON t.TicketTypeSkey = tk.tickettypeskey
        CROSS JOIN month_bounds mb
        WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
          AND t.TicketFLAG = 'Yes'
          AND d.datefull BETWEEN mb.month_start AND mb.month_end
    ),

    -- Format classification (top-tier mapping wins; else 3D; else Regular)
    -- DBox is evaluated independently below -- not part of this waterfall
    ticket_events_classified AS (
        SELECT
            *,
            CASE
                WHEN auditorium_experience_id = 2      THEN 'AVX'
                WHEN auditorium_experience_id = 3      THEN 'IMAX'
                WHEN auditorium_experience_id = 4      THEN 'VIP'
                WHEN auditorium_experience_id IN (5,6) THEN 'ALT'
                WHEN auditorium_experience_id = 7      THEN 'FourDX'
                WHEN auditorium_experience_id = 8      THEN 'ScreenX'
                WHEN performance_experience_id = 2     THEN 'ThreeD'
                ELSE 'Regular'
            END AS final_format
        FROM ticket_events
    ),

    -- ----------------------------------------------------------------------------
    -- Per-population ticket aggregation
    -- tickets_free_redeemed / companion_tickets_redeemed / the *_999 / *_1099
    -- sub-splits are CC-only concepts -- set NULL here for Scene/Total_Biz, not zero
    -- CC and Scene apply CDE_ID IS NOT NULL explicitly; Total_Biz does not.
    -- ----------------------------------------------------------------------------
    tickets_agg AS (
        SELECT
            'CC' AS population_type,
            COUNT(DISTINCT cde_id)                                                                     AS distinct_members,
            COUNT(DISTINCT cde_id, datefull)                                                           AS total_visits,
            SUM(CASE WHEN final_format = 'Regular' THEN quantity ELSE 0 END)                          AS tickets_regular,
            SUM(CASE WHEN final_format != 'Regular' THEN quantity ELSE 0 END)                          AS tickets_upgrades,
            SUM(CASE WHEN is_dbox_flag = 'Yes' THEN quantity ELSE 0 END)                               AS tickets_dbox,
            SUM(CASE WHEN final_format = 'AVX' THEN quantity ELSE 0 END)                               AS tickets_avx,
            SUM(CASE WHEN final_format = 'IMAX' THEN quantity ELSE 0 END)                              AS tickets_imax,
            SUM(CASE WHEN final_format = 'VIP' THEN quantity ELSE 0 END)                               AS tickets_vip,
            SUM(CASE WHEN final_format = 'ALT' THEN quantity ELSE 0 END)                               AS tickets_alt,
            SUM(CASE WHEN final_format = 'FourDX' THEN quantity ELSE 0 END)                            AS tickets_fourdx,
            SUM(CASE WHEN final_format = 'ScreenX' THEN quantity ELSE 0 END)                           AS tickets_screenx,
            SUM(CASE WHEN final_format = 'ThreeD' THEN quantity ELSE 0 END)                            AS tickets_threed,
            SUM(CASE WHEN paid_flag IS DISTINCT FROM 'Yes' THEN quantity ELSE 0 END)                   AS tickets_free,
            SUM(CASE WHEN paid_flag = 'Yes' THEN quantity ELSE 0 END)                                  AS tickets_paid,
            SUM(CASE WHEN discount_type_id IN (6,8) THEN quantity ELSE 0 END)                          AS tickets_free_redeemed,
            SUM(CASE WHEN discount_type_id IN (5,7) THEN quantity ELSE 0 END)                          AS companion_tickets_redeemed,
            SUM(CASE WHEN discount_type_id = 1 THEN quantity ELSE 0 END)                               AS tickets_undiscounted,
            SUM(CASE WHEN discount_type_id = 6 THEN quantity ELSE 0 END)                               AS tickets_free_999,
            SUM(CASE WHEN discount_type_id = 8 THEN quantity ELSE 0 END)                               AS tickets_free_1099,
            SUM(CASE WHEN discount_type_id = 5 THEN quantity ELSE 0 END)                               AS tickets_companion_999,
            SUM(CASE WHEN discount_type_id = 7 THEN quantity ELSE 0 END)                               AS tickets_companion_1099
        FROM ticket_events_classified
        WHERE cineclub_flag = 'Yes' AND cde_id IS NOT NULL

        UNION ALL

        SELECT
            'Scene' AS population_type,
            COUNT(DISTINCT cde_id),
            COUNT(DISTINCT cde_id, datefull),
            SUM(CASE WHEN final_format = 'Regular' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format != 'Regular' THEN quantity ELSE 0 END),
            SUM(CASE WHEN is_dbox_flag = 'Yes' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'AVX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'IMAX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'VIP' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ALT' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'FourDX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ScreenX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ThreeD' THEN quantity ELSE 0 END),
            SUM(CASE WHEN paid_flag IS DISTINCT FROM 'Yes' THEN quantity ELSE 0 END),
            SUM(CASE WHEN paid_flag = 'Yes' THEN quantity ELSE 0 END),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT)
        FROM ticket_events_classified
        -- Scene = Non-CC Scene members only (CC members excluded)
        WHERE scene_flag = 'Yes' AND cineclub_flag IS DISTINCT FROM 'Yes' AND cde_id IS NOT NULL

        UNION ALL

        SELECT
            'Total_Biz' AS population_type,
            COUNT(DISTINCT cde_id),
            COUNT(DISTINCT cde_id, datefull),
            SUM(CASE WHEN final_format = 'Regular' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format != 'Regular' THEN quantity ELSE 0 END),
            SUM(CASE WHEN is_dbox_flag = 'Yes' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'AVX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'IMAX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'VIP' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ALT' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'FourDX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ScreenX' THEN quantity ELSE 0 END),
            SUM(CASE WHEN final_format = 'ThreeD' THEN quantity ELSE 0 END),
            SUM(CASE WHEN paid_flag IS DISTINCT FROM 'Yes' THEN quantity ELSE 0 END),
            SUM(CASE WHEN paid_flag = 'Yes' THEN quantity ELSE 0 END),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT),
            CAST(NULL AS BIGINT)
        FROM ticket_events_classified
        -- No CDE_ID filter: Total_Biz includes anonymous/unidentified ticket buyers
    ),

    -- ----------------------------------------------------------------------------
    -- MOM-matched member/visit grain (CC + Scene only; Total_Biz keeps tickets_agg).
    -- Any theatre purchase (tickets AND concessions), netted per
    -- cde_id x day x location, kept if net quantity > 0 and cde_id > 0.
    -- CC = CineClubFLAG 'Yes'; Scene = SCENEFLAG 'Yes' AND not CC (Non-CC Scene).
    -- Cohorts are mutually exclusive.
    -- Visits = distinct cde_id x day.
    -- ----------------------------------------------------------------------------
    member_visit_grain AS (
        SELECT population_type, cde_id, datefull
        FROM (
            SELECT
                explode(filter(array(
                    CASE WHEN t.CineClubFLAG = 'Yes' THEN 'CC' END,
                    CASE WHEN t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes' THEN 'Scene' END
                ), p -> p IS NOT NULL))             AS population_type,
                t.CDE_ID                            AS cde_id,
                d.datefull                          AS datefull,
                t.LocationId                        AS location_id,
                t.Quantity                          AS quantity
            FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
            JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc
                ON t.LocationId = loc.LocationID
            JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d
                ON t.VisitDateId = d.dateid
            CROSS JOIN month_bounds mb
            WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
              AND t.CDE_ID IS NOT NULL
              AND d.datefull BETWEEN mb.month_start AND mb.month_end
        ) x
        WHERE population_type IS NOT NULL
        GROUP BY population_type, cde_id, datefull, location_id
        HAVING SUM(quantity) > 0 AND cde_id > 0
    ),

    member_visits_agg AS (
        SELECT
            population_type,
            COUNT(DISTINCT cde_id)            AS distinct_members,
            COUNT(DISTINCT cde_id, datefull)  AS total_visits
        FROM member_visit_grain
        GROUP BY population_type
    ),

    -- ----------------------------------------------------------------------------
    -- Credited -- event-based, all plans; 9.99/10.99 split by rate (monthly
    -- subtotal 9.99/10.99, or annual subtotal 109.89/119.88 vs 120.89).
    -- gift-plan and stray null-subtotal rows count toward the total but not
    -- toward either tier. In-market stays on the month-end snapshot.
    -- ----------------------------------------------------------------------------
    credit_cycles AS (
        SELECT DISTINCT
            cc_user_profile_id,
            current_term_start_datetime,
            subtotal
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        CROSS JOIN month_bounds mb
        WHERE event_type IN ('payment_succeeded', 'subscription_renewed')
          AND current_term_start_datetime >= mb.month_start
          AND current_term_start_datetime < ADD_MONTHS(mb.month_start, 1)
    ),

    snapshot_current_members AS (
        SELECT
            cc_user_profile_id,
            recognitions_left,
            recognitions_left_10_99
        FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot sn
        CROSS JOIN month_bounds mb
        WHERE sn.snapshot_date_id = DATE_FORMAT(mb.month_end, 'yyyyMMdd')
    ),

    recognitions AS (
        SELECT
            (SELECT COUNT(*) FROM credit_cycles)                                                         AS tickets_credited,
            (SELECT COUNT(*) FROM credit_cycles WHERE subtotal IN (9.99, 109.89, 119.88))                 AS tickets_credited_999,
            (SELECT COUNT(*) FROM credit_cycles WHERE subtotal IN (10.99, 120.89))                        AS tickets_credited_1099,
            SUM(cur.recognitions_left)                                                                    AS tickets_in_market_999,
            SUM(COALESCE(cur.recognitions_left_10_99, 0))                                                 AS tickets_in_market_1099
        FROM snapshot_current_members cur
    )

    -- ============================================================================
    -- Final assembly
    -- ============================================================================
    SELECT
        mb.month_start AS report_month,
        ta.population_type,
        COALESCE(mv.distinct_members, ta.distinct_members) AS distinct_members,
        COALESCE(mv.total_visits, ta.total_visits)         AS total_visits,
        ta.tickets_regular,
        ta.tickets_upgrades,
        ta.tickets_dbox,
        ta.tickets_avx,
        ta.tickets_imax,
        ta.tickets_vip,
        ta.tickets_alt,
        ta.tickets_fourdx,
        ta.tickets_screenx,
        ta.tickets_threed,
        ta.tickets_free,
        ta.tickets_paid,
        ta.tickets_free_redeemed,
        ta.companion_tickets_redeemed,
        ta.tickets_undiscounted,
        CASE WHEN ta.population_type = 'CC' THEN ta.tickets_free_999      / NULLIF(ta.tickets_free_999 + ta.tickets_free_1099, 0)           ELSE NULL END AS share_9_99_Free,
        CASE WHEN ta.population_type = 'CC' THEN ta.tickets_free_1099     / NULLIF(ta.tickets_free_999 + ta.tickets_free_1099, 0)           ELSE NULL END AS share_10_99_Free,
        CASE WHEN ta.population_type = 'CC' THEN ta.tickets_companion_999 / NULLIF(ta.tickets_companion_999 + ta.tickets_companion_1099, 0) ELSE NULL END AS share_9_99_Companion,
        CASE WHEN ta.population_type = 'CC' THEN ta.tickets_companion_1099/ NULLIF(ta.tickets_companion_999 + ta.tickets_companion_1099, 0) ELSE NULL END AS share_10_99_Companion,
        CASE WHEN ta.population_type = 'CC' THEN r.tickets_credited                                                                                    ELSE NULL END AS tickets_credited,
        CASE WHEN ta.population_type = 'CC' THEN r.tickets_in_market_999 + COALESCE(r.tickets_in_market_1099, 0) ELSE NULL END AS tickets_in_market,
        CASE WHEN ta.population_type = 'CC' THEN r.tickets_credited_999   / NULLIF(r.tickets_credited_999 + r.tickets_credited_1099, 0)                ELSE NULL END AS share_9_99_Credited,
        CASE WHEN ta.population_type = 'CC' THEN r.tickets_credited_1099  / NULLIF(r.tickets_credited_999 + r.tickets_credited_1099, 0)                ELSE NULL END AS share_10_99_Credited,
        CASE WHEN ta.population_type = 'CC' THEN r.tickets_in_market_999  / NULLIF(r.tickets_in_market_999 + COALESCE(r.tickets_in_market_1099, 0), 0)  ELSE NULL END AS share_9_99_InMarket,
        CASE WHEN ta.population_type = 'CC' THEN COALESCE(r.tickets_in_market_1099, 0) / NULLIF(r.tickets_in_market_999 + COALESCE(r.tickets_in_market_1099, 0), 0)  ELSE NULL END AS share_10_99_InMarket
    FROM tickets_agg ta
    LEFT JOIN member_visits_agg mv
        ON mv.population_type = ta.population_type
    CROSS JOIN month_bounds mb
    CROSS JOIN recognitions r
    ORDER BY ta.population_type;

    SET v_report_month = add_months(v_report_month, 1);
  END WHILE;
END;